# Day 3 — ILT 3: Change Data Feed (CDF) in Databricks
**Time:** 4:00 PM – 4:30 PM  
**What comes next:** Hands-on — Enable CDF, query `table_changes()` (4:30–5:30 PM)  

---
### What we cover today
1. What is CDF and how it is different from source CDC (WAL)
2. How to enable CDF on a Delta table
3. Reading changes using `table_changes()`
4. Real use case — propagate Bronze changes to Silver automatically

> **Instructor note:** 30 minutes only. Keep it tight. ~10 min CDF theory, ~15 min live code, ~5 min wrap-up. Students just came from a 2-hour hands-on so keep energy high with quick demos.

## Section 1 — CDF vs Source CDC — What is the Difference?

We just learned about CDC from Supabase (source-level). CDF is a different but related concept.

| | Source CDC (WAL) | Databricks CDF |
|-|-----------------|----------------|
| **Where** | Inside PostgreSQL (Supabase) | Inside a Delta table on ADLS |
| **What it tracks** | Changes in the source database | Changes to a Delta table |
| **Triggered by** | Any app writing to the DB | Any Spark write to the Delta table |
| **Use case** | Capture changes from Supabase → Bronze | Propagate changes from Bronze → Silver |
| **Technology** | PostgreSQL WAL + logical replication | `delta.enableChangeDataFeed = true` |

### How they work TOGETHER in GlobalMart

```
Customer updates their city in the app
         |
         v
Supabase WAL (source CDC) captures the UPDATE
         |
         v
JDBC polling → Bronze Delta table updated
         |
         v  ← THIS is where CDF kicks in
CDF detects the change in Bronze
         |
         v
Silver pipeline reads only the changed rows from Bronze
         |
         v
Silver customers table is updated with the new city
```

> **Source CDC** = capturing changes FROM your database  
> **CDF** = propagating changes BETWEEN your Delta tables

## Section 2 — What CDF Gives You

When CDF is enabled, every write to a Delta table also writes a **change log** alongside the data.

```
bronze/customers/
    _delta_log/              ← regular Delta log (always exists)
    _change_data/            ← CDF change log (only when CDF is enabled)
        part-00000.parquet   ← contains INSERT/UPDATE/DELETE records
    part-00000.parquet       ← actual current data
```

### What a CDF record looks like

| customer_id | customer_city | _change_type | _commit_version | _commit_timestamp |
|-------------|--------------|--------------|-----------------|-------------------|
| C001 | Mumbai | insert | 1 | 2024-01-10 09:00 |
| C001 | Delhi | update_postimage | 3 | 2024-02-15 14:30 |
| C001 | Mumbai | update_preimage | 3 | 2024-02-15 14:30 |
| C999 | Chennai | delete | 5 | 2024-03-01 10:00 |

**`_change_type` values:**
- `insert` — new row added
- `update_preimage` — what the row looked like BEFORE the update
- `update_postimage` — what the row looks like AFTER the update
- `delete` — row that was deleted

## Setup

In [ ]:
# ─── ADLS Setup ───────────────────────────────────────────────────────────────
storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← Replace

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

raw_path    = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw"
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"
silver_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

print("Connected to ADLS!")

## Section 3 — Enabling CDF on a Delta Table

In [ ]:
# ─── Step 1: Write a Delta table with CDF enabled ─────────────────────────────
# Two ways to enable CDF:
# Option A — when CREATING the table (add the property)
# Option B — on an EXISTING table (ALTER TABLE)

cdf_demo_path = f"{bronze_path}/customers_cdf_demo"

# Read customers from raw CSV
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/customers.csv")

# Option A: Write with CDF enabled from the start
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(cdf_demo_path)

print(f"Delta table written with CDF enabled: {customers_df.count():,} rows")

# Verify CDF is enabled
cdf_props = spark.sql(f"DESCRIBE DETAIL delta.`{cdf_demo_path}`")
cdf_props.select("properties").show(truncate=False)

In [ ]:
# ─── Option B: Enable CDF on an EXISTING table ───────────────────────────────
# If you already have a Delta table and want to turn on CDF

spark.sql(f"""
    ALTER TABLE delta.`{cdf_demo_path}`
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("CDF enabled on existing table via ALTER TABLE")

In [ ]:
# ─── Step 2: Simulate changes — write some updates ────────────────────────────
# We need to make some changes to the table so CDF has something to track
# Simulate: a batch of customers changed their city

from pyspark.sql.functions import col, when, lit
from delta.tables import DeltaTable

# Load the Delta table
delta_customers = DeltaTable.forPath(spark, cdf_demo_path)

# Simulate: first 50 customers moved to a new city
# In real life this would come from a Supabase CDC feed
top50_ids = customers_df.limit(50).select("CustomerID").collect()
id_list   = [row.CustomerID for row in top50_ids]

update_df = customers_df.limit(50).withColumn("City", lit("Updated City"))

# MERGE — update existing rows where CustomerID matches
delta_customers.alias("existing") \
    .merge(
        update_df.alias("changes"),
        "existing.CustomerID = changes.CustomerID"
    ) \
    .whenMatchedUpdateAll() \
    .execute()

print("Simulated update: 50 customers' cities changed")
print("CDF has captured this change — let us read it next")

## Section 4 — Reading Changes with `table_changes()`

In [ ]:
# ─── Read CDF changes using table_changes() ───────────────────────────────────
# table_changes() reads the _change_data folder
# You specify a starting version — it returns everything AFTER that version

# Read changes starting from version 1 (after the first write)
changes_df = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 1) \
    .load(cdf_demo_path)

print(f"Total change records captured: {changes_df.count():,}")
print()

# Show the change types breakdown
print("Change types:")
changes_df.groupBy("_change_type").count().show()

print("Sample change records:")
changes_df.select(
    "CustomerID", "City", "_change_type", "_commit_version", "_commit_timestamp"
).show(10, truncate=False)

In [ ]:
# ─── Keep only the latest version of each changed row ─────────────────────────
# From CDF, we get both pre-image and post-image for updates
# For propagating to Silver, we only want the post-image (latest state)

from pyspark.sql.functions import col

# Filter: keep only inserts and update_postimage (the new values)
# Discard update_preimage (the old values — we don't want to re-write stale data)
latest_changes = changes_df.filter(
    col("_change_type").isin(["insert", "update_postimage"])
).drop("_change_type", "_commit_version", "_commit_timestamp")

print(f"Latest change records to propagate: {latest_changes.count():,}")
latest_changes.show(5, truncate=True)

In [ ]:
# ─── Propagate changes from Bronze → Silver using MERGE ───────────────────────
# This is the full CDF pipeline:
#   1. Read changes from Bronze CDF (only rows that changed)
#   2. MERGE into Silver (update existing, insert new)
# Result: Silver is always up to date, without reloading the full table

silver_customers_path = f"{silver_path}/customers"

# First, make sure Silver exists (write all customers if it doesn't)
try:
    silver_count = spark.read.format("delta").load(silver_customers_path).count()
    print(f"Silver customers exists: {silver_count:,} rows")
except:
    customers_df.write.format("delta").mode("overwrite").save(silver_customers_path)
    print("Silver customers created from full load")

# MERGE the CDF changes into Silver
silver_delta = DeltaTable.forPath(spark, silver_customers_path)

silver_delta.alias("silver") \
    .merge(
        latest_changes.alias("changes"),
        "silver.CustomerID = changes.CustomerID"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"Silver updated via CDF propagation!")
print(f"Silver customers now: {spark.read.format('delta').load(silver_customers_path).count():,} rows")

## Section 5 — CDF in Production: The Complete Pattern

```python
# ── The complete Bronze → Silver CDC relay pipeline ──

# Step 1: Get the last version we processed (stored in a control table)
last_processed_version = get_last_processed_version("customers")  # e.g., 5

# Step 2: Read only changes AFTER last processed version
new_changes = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", last_processed_version + 1) \
    .load(bronze_customers_path)

# Step 3: Keep only post-images (latest state of changed rows)
to_upsert = new_changes.filter(col("_change_type").isin(["insert", "update_postimage"]))

# Step 4: MERGE into Silver
DeltaTable.forPath(spark, silver_customers_path).alias("s") \
    .merge(to_upsert.alias("c"), "s.CustomerID = c.CustomerID") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

# Step 5: Save the current version as the new watermark
save_last_processed_version("customers", current_version)
```

> **This pattern is what we build in Week 3 (Day 11 ILT 3: CDF-Based Incremental Loading).**  
> Today is the introduction — understand the concept before we implement it fully.

## Recap

| Topic | Key Takeaway |
|-------|--------------|
| CDF vs Source CDC | Source CDC = captures changes from Supabase. CDF = tracks changes inside Delta tables |
| Enabling CDF | `delta.enableChangeDataFeed = true` — set when writing or via ALTER TABLE |
| `_change_data` folder | Created automatically — stores change records alongside data |
| `_change_type` | insert / update_preimage / update_postimage / delete |
| Reading CDF | `option("readChangeFeed", "true").option("startingVersion", N)` |
| Use case | Bronze → Silver propagation — only process rows that changed |

---

## Hands-on Next (4:30 PM — 1 hour)

1. Enable CDF on your Bronze `customers` table
2. Write some updates to the table (simulate city changes)
3. Read changes with `table_changes()` — see insert / update_postimage / update_preimage
4. Filter to post-images only and MERGE into Silver
5. Verify Silver is updated with the new city values

---

## Tomorrow — Day 4
- **ILT 1 (9:30 AM):** Schema Evolution — what happens when a new column is added to the source
- **ILT 2 (12:00 PM):** Recap — Ingestion Patterns Across All 4 Sources
- **Hands-on:** Mini end-to-end — land one record from each of the 4 sources into Bronze